# Data stationarity assumptions

ARIMA/SARIMAX assumes the time series to be stationary both in mean and variance.

In other words, the series should have constant mean and variance over time.

- Log transformation to stabilize variance.
- Differencing to stabilize mean.

In [1]:
from modules import utils
utils.configure_plotly_template()


## Load data

In [2]:
path = '../../../data/statsmodels/AirPassengers.parquet'


In [3]:
import pandas as pd
df = pd.read_parquet(path).asfreq("ME")
df.columns = ['values']

df

,values
1949-01-31,112
1949-02-28,118
...,...
1960-11-30,390
1960-12-31,432


## Achieve stationarity

| **Property**             | **Issue**           | **Fix**               | **Test** |
| ------------------------ | ------------------- | --------------------- | -------- |
| Stationarity in variance | Changing volatility | Log transform / GARCH | ARCH     |
| Stationarity in mean     | Trend / Drift       | Differencing          | ADF      |


### Variance

In [4]:
df['values'].plot()

In [5]:
import numpy as np
df['values_log'] = np.log(df['values'])

df

,values,values_log
1949-01-31,112,4.718499
1949-02-28,118,4.770685
...,...,...
1960-11-30,390,5.966147
1960-12-31,432,6.068426


In [6]:
fig = df.plot(facet_col='variable', facet_col_spacing=0.1)
fig.update_yaxes(matches=None)
fig.update_layout(width=1200)


for attr in dir(fig.layout):
    if attr.startswith("yaxis"):
        axis = getattr(fig.layout, attr)
        if axis:
            axis.showticklabels = True
        
fig


### Mean

In [7]:
from statsmodels.tsa.stattools import adfuller
adfuller(df['values_log'])

(np.float64(-1.7170170891069683),
 np.float64(0.4223667747703874),
 13,
 130,
 {'1%': np.float64(-3.4816817173418295),
  '5%': np.float64(-2.8840418343195267),
  '10%': np.float64(-2.578770059171598)},
 np.float64(-445.3990312497209))

In [8]:
df['values_log_diff'] = df['values_log'].diff()
df.dropna(inplace=True)
df

,values,values_log,values_log_diff
1949-02-28,118,4.770685,0.052186
1949-03-31,132,4.882802,0.112117
...,...,...,...
1960-11-30,390,5.966147,-0.167251
1960-12-31,432,6.068426,0.102279


In [10]:
adfuller(df['values_log_diff'], maxlag=12)

(np.float64(-3.0530320109154845),
 np.float64(0.030229987648694875),
 12,
 130,
 {'1%': np.float64(-3.4816817173418295),
  '5%': np.float64(-2.8840418343195267),
  '10%': np.float64(-2.578770059171598)},
 np.float64(-448.718127711991))

In [11]:
fig = df.plot(facet_col='variable', facet_col_spacing=0.1)
fig.update_yaxes(matches=None)
fig.update_layout(width=1200)


for attr in dir(fig.layout):
    if attr.startswith("yaxis"):
        axis = getattr(fig.layout, attr)
        if axis:
            axis.showticklabels = True
        
fig

## Modelling

### Model fit

In [12]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
model = SARIMAX(df['values_log'], order=(1, 1, 1), seasonal_order=(0, 1, 1, 12))
model_fit = model.fit()

In [13]:
model_fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                     SARIMAX Results                                      
==========================================================================================
Dep. Variable:                         values_log   No. Observations:                  143
Model:             SARIMAX(1, 1, 1)x(0, 1, 1, 12)   Log Likelihood                 242.580
Date:                            Sat, 30 Aug 2025   AIC                           -477.159
Time:                                    20:08:48   BIC                           -465.689
Sample:                                02-28-1949   HQIC                          -472.499
                                     - 12-31-1960                                         
Covariance Type:                              opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.2003      0.198      1.010      0.312      -0.188       0.589
ma.L1         -0.5833      0.173     -3.376      0.001      -0.922      -0.245
ma.S.L12      -0.5614      0.106     -5.285      0.000      -0.770      -0.353
sigma2         0.0014      0.000      8.581      0.000       0.001       0.002
===================================================================================
Ljung-Box (L1) (Q):                   0.04   Jarque-Bera (JB):                 3.53
Prob(Q):                              0.83   Prob(JB):                         0.17
Heteroskedasticity (H):               0.60   Skew:                             0.02
Prob(H) (two-sided):                  0.10   Kurtosis:                         3.81
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

### Forecast

In [14]:
forecast = model_fit.forecast(steps=48)

In [15]:
df_forecast = pd.DataFrame({
    'historical': df['values'],
    'forecast': np.exp(forecast)
})

df_forecast

,historical,forecast
1949-02-28,118.0,NaN
1949-03-31,132.0,NaN
...,...,...
1964-11-30,NaN,577.572384
1964-12-31,NaN,641.262027


In [16]:
df_forecast.plot()